# LeetCode 1487 — Making File Names Unique bằng Python

Link đề: https://leetcode.com/problems/making-file-names-unique/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def getFolderNames(self, names: list[str]) -> list[str]:
        ...
```

## 1. Mục tiêu học

1. Mô phỏng đúng quy tắc đặt tên của hệ điều hành: tên trùng ⇒ thêm hậu tố `(k)` với `k` **nhỏ nhất** còn trống.
2. Thấy rõ vì sao **tên mới sinh ra cũng phải được ghi nhận** (nó có thể đụng với tên nhập sau này).
3. Đi từ `list` (tra `O(n)`) → `set` + thử `k` từ 1 (tệ nhất `O(n²)`) → `dict` nhớ **`k` kế tiếp** cho từng tên gốc (khấu hao `O(n·L)`).
4. Hiểu phân tích **khấu hao** (amortized): tổng số lần tăng `k` trên toàn bộ bài bị chặn bởi `n`.
5. Tránh các bẫy: đếm số lần xuất hiện thay vì kiểm tra trống, tự ý "bóc" hậu tố của tên nhập vào, quên ghi tên mới.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `names` độ dài `n`. Ở phút thứ `i` ta tạo một thư mục tên `names[i]`. Hai thư mục không được trùng tên: nếu tên đã được dùng, hệ thống thêm hậu tố dạng `(k)`, với `k` là **số nguyên dương nhỏ nhất** sao cho tên mới vẫn chưa ai dùng. Trả về mảng `ans` với `ans[i]` là tên thực tế của thư mục thứ `i`.

**Ví dụ 1:** `["pes","fifa","gta","pes(2019)"]` → `["pes","fifa","gta","pes(2019)"]` (không trùng gì; `pes(2019)` chỉ là một tên bình thường).

**Ví dụ 2:** `["gta","gta(1)","gta","avalon"]` → `["gta","gta(1)","gta(2)","avalon"]` (`gta` thứ hai: `gta(1)` đã có ⇒ `k = 2`).

**Ví dụ 3:** `["onepiece","onepiece(1)","onepiece(2)","onepiece(3)","onepiece"]` → `["onepiece","onepiece(1)","onepiece(2)","onepiece(3)","onepiece(4)"]`.

**Ràng buộc:** `1 <= n <= 5·10^4`; `1 <= len(names[i]) <= 20`; chỉ gồm chữ thường, chữ số và dấu ngoặc tròn.

In [1]:
import random
import time
from collections import defaultdict
from typing import List

examples = [
    (["pes", "fifa", "gta", "pes(2019)"], ["pes", "fifa", "gta", "pes(2019)"]),
    (["gta", "gta(1)", "gta", "avalon"], ["gta", "gta(1)", "gta(2)", "avalon"]),
    (["onepiece", "onepiece(1)", "onepiece(2)", "onepiece(3)", "onepiece"],
     ["onepiece", "onepiece(1)", "onepiece(2)", "onepiece(3)", "onepiece(4)"]),
]
for names, want in examples:
    print(names, "->", want)

['pes', 'fifa', 'gta', 'pes(2019)'] -> ['pes', 'fifa', 'gta', 'pes(2019)']
['gta', 'gta(1)', 'gta', 'avalon'] -> ['gta', 'gta(1)', 'gta(2)', 'avalon']
['onepiece', 'onepiece(1)', 'onepiece(2)', 'onepiece(3)', 'onepiece'] -> ['onepiece', 'onepiece(1)', 'onepiece(2)', 'onepiece(3)', 'onepiece(4)']


## 3. Kiến thức nền

### 3.1 f-string để ghép tên

`f"{name}({k})"` tạo chuỗi `name(k)`. Chú ý số `k` được in **không có số 0 đứng đầu**.

In [2]:
name, k = "gta", 2
print(f"{name}({k})")
print([f"doc({k})" for k in range(1, 4)])

gta(2)
['doc(1)', 'doc(2)', 'doc(3)']


### 3.2 `list` vs `set`: chi phí kiểm tra `x in ...`

- `x in list` duyệt tuần tự: `O(n)`.
- `x in set` / `x in dict` dùng bảng băm: `O(1)` trung bình (cộng chi phí băm chuỗi `O(L)`).

In [3]:
data_list = [f"f{i}" for i in range(200000)]
data_set = set(data_list)
t0 = time.perf_counter(); _ = "f199999" in data_list; t1 = time.perf_counter()
_ = "f199999" in data_set; t2 = time.perf_counter()
print(f"list: {(t1 - t0) * 1e6:.1f} µs | set: {(t2 - t1) * 1e6:.1f} µs")

list: 2247.5 µs | set: 47.5 µs


### 3.3 `dict` lưu thông tin kèm theo khoá

`d[name] = k` vừa đánh dấu "tên đã dùng" (khoá có trong `d`), vừa lưu "lần sau thử `k` từ đâu". `d.get(key, default)` trả giá trị mặc định khi chưa có khoá.

In [4]:
d = {}
d["gta"] = 1
print("gta" in d, d.get("gta"), d.get("fifa", "chưa có"))
d["gta"] += 1
print(d)

True 1 chưa có
{'gta': 2}


### 3.4 Vòng `while` tìm giá trị nhỏ nhất thoả điều kiện

Mẫu "thử tăng dần đến khi được": `k = start; while điều_kiện_xấu(k): k += 1`. Câu hỏi then chốt luôn là: **vòng này chạy tổng cộng bao nhiêu lần?**

In [5]:
used = {"a", "a(1)", "a(2)", "a(4)"}
k = 1
while f"a({k})" in used:
    k += 1
print("k nhỏ nhất trống:", k)

k nhỏ nhất trống: 3


## 4. Ý tưởng nền tảng / trực giác

Làm tay **Ví dụ 2** `["gta","gta(1)","gta","avalon"]`, giữ một tập `used`:

| i | names[i] | đã dùng? | thử k | ans[i] | used sau bước |
|---|---|---|---|---|---|
| 0 | gta | không | — | gta | {gta} |
| 1 | gta(1) | không | — | gta(1) | {gta, gta(1)} |
| 2 | gta | **có** | k=1: gta(1) có ✗; k=2: gta(2) trống ✓ | gta(2) | {…, gta(2)} |
| 3 | avalon | không | — | avalon | {…, avalon} |

Hai nhận xét quan trọng:
1. Tên nhập vào như `gta(1)` **không** được coi là "phiên bản của gta" khi nhập — nó chỉ là một chuỗi. Nhưng nó **chiếm chỗ** của `gta(1)`.
2. Tên **sinh ra** (`gta(2)`) cũng phải vào `used`, vì sau này có thể có ai nhập đúng `gta(2)`.

In [6]:
names = ["gta", "gta(1)", "gta", "avalon"]
used = set(); ans = []
for nm in names:
    if nm not in used:
        res = nm
    else:
        k = 1
        while f"{nm}({k})" in used:
            k += 1
        res = f"{nm}({k})"
    used.add(res); ans.append(res)
    print(f"{nm:8} -> {res:8} | used = {sorted(used)}")

gta      -> gta      | used = ['gta']
gta(1)   -> gta(1)   | used = ['gta', 'gta(1)']
gta      -> gta(2)   | used = ['gta', 'gta(1)', 'gta(2)']
avalon   -> avalon   | used = ['avalon', 'gta', 'gta(1)', 'gta(2)']


## 5. Cách 1 — Brute Force / baseline: dùng `list` và thử `k` từ 1

**Ý tưởng:** đúng như mô tả đề. Giữ danh sách `ans` các tên đã cấp; với mỗi tên, kiểm tra trong `ans` (tra tuyến tính), nếu trùng thì thử `k = 1, 2, …`.

In [7]:
def folder_names_brute(names):
    ans = []
    for nm in names:
        if nm not in ans:                 # O(len(ans))
            ans.append(nm)
            continue
        k = 1
        while f"{nm}({k})" in ans:        # mỗi lần thử O(len(ans))
            k += 1
        ans.append(f"{nm}({k})")
    return ans

for names, want in examples:
    print(folder_names_brute(names) == want, folder_names_brute(names))

True ['pes', 'fifa', 'gta', 'pes(2019)']
True ['gta', 'gta(1)', 'gta(2)', 'avalon']
True ['onepiece', 'onepiece(1)', 'onepiece(2)', 'onepiece(3)', 'onepiece(4)']


**Độ phức tạp:** mỗi phép `in ans` tốn `O(n)`; số lần thử `k` cho một tên có thể tới `O(n)` ⇒ tệ nhất **`O(n³)`** phép so chuỗi (ví dụ `n` tên giống hệt nhau: tên thứ `i` thử `i` giá trị `k`, mỗi giá trị dò `i` phần tử). Với `n = 5·10^4` là bất khả thi.

**Nút thắt:** (1) tra cứu tuyến tính; (2) với mỗi lần trùng lại thử `k` **từ đầu**.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — sửa nút thắt (1):** thay `list` bằng `set` ⇒ tra `O(1)`. Còn lại nút thắt (2): `n` tên `"a"` giống nhau ⇒ tên thứ `i` vẫn thử `k = 1..i` ⇒ `O(n²)`.

**Bước 2 — sửa nút thắt (2):** ghi nhớ cho từng tên gốc `nm` giá trị `next_k[nm]` = "lần trước đã dùng tới `k` nào, lần sau thử tiếp từ đó".

**Vì sao được phép bắt đầu từ `next_k[nm]` thay vì 1?** Tập tên đã dùng **chỉ tăng, không bao giờ giảm**. Nếu lần trước ta đã kiểm tra `nm(1)…nm(k−1)` đều bị chiếm, thì bây giờ chúng **vẫn bị chiếm** ⇒ không cần thử lại.

**Invariant:** với mọi tên `nm` đã có trong `d`, mọi `nm(j)` với `1 ≤ j < d[nm]` đều đã được dùng.

**Quy tắc:**
- Nếu `nm` chưa có trong `d` ⇒ dùng `nm`, đặt `d[nm] = 1`.
- Nếu có ⇒ `k = d[nm]`; trong khi `nm(k)` có trong `d` thì `k += 1`; cấp `new = nm(k)`; đặt `d[nm] = k + 1` và `d[new] = 1` (tên mới cũng là một tên gốc tiềm năng!).

Một `dict` duy nhất làm hai việc: khoá = "tên đã dùng", giá trị = "`k` kế tiếp nên thử".

## 7. Các cách tiếp theo

### Cách 2 — `set` + thử `k` từ 1

Giống Cách 1 nhưng tra cứu bằng `set`. Đúng, đơn giản, nhưng tệ nhất `O(n²)` khi có nhiều tên trùng.

In [8]:
def folder_names_set(names):
    used = set()
    ans = []
    for nm in names:
        res = nm
        if nm in used:
            k = 1
            while f"{nm}({k})" in used:
                k += 1
            res = f"{nm}({k})"
        used.add(res)
        ans.append(res)
    return ans

for names, want in examples:
    print(folder_names_set(names) == want)

True
True
True


Code nộp LeetCode (Cách 2 — có thể TLE với test xấu nhất như `["a"] * 50000`):

In [9]:
class SolutionSet:
    def getFolderNames(self, names: list[str]) -> list[str]:
        used = set()
        ans = []
        for nm in names:
            res = nm
            if nm in used:
                k = 1
                while f"{nm}({k})" in used:
                    k += 1
                res = f"{nm}({k})"
            used.add(res)
            ans.append(res)
        return ans

print(SolutionSet().getFolderNames(["a", "a", "a(1)", "a"]))

['a', 'a(1)', 'a(1)(1)', 'a(2)']


**Độ phức tạp:** `O(n² · L)` tệ nhất (`L ≤ 20` + độ dài số `k`), `O(n · L)` bộ nhớ.

### Cách 3 — `dict` nhớ `k` kế tiếp (tối ưu)

Giải thích từng đoạn:
1. `d = {}`: khoá là tên đã dùng, giá trị là `k` nên thử lần sau cho tên gốc đó.
2. Tên mới hoàn toàn ⇒ `d[nm] = 1`.
3. Tên trùng ⇒ tiếp tục từ `d[nm]`, nhảy qua các `nm(k)` đã bị chiếm.
4. Ghi nhận cả `d[nm] = k + 1` (lần sau khỏi thử lại `k`) và `d[new] = 1`.

In [10]:
def folder_names_dict(names):
    d = {}
    ans = []
    for nm in names:
        if nm not in d:
            d[nm] = 1
            ans.append(nm)
            continue
        k = d[nm]
        while f"{nm}({k})" in d:
            k += 1
        new = f"{nm}({k})"
        d[nm] = k + 1
        d[new] = 1
        ans.append(new)
    return ans

for names, want in examples:
    print(folder_names_dict(names) == want, folder_names_dict(names))

True ['pes', 'fifa', 'gta', 'pes(2019)']
True ['gta', 'gta(1)', 'gta(2)', 'avalon']
True ['onepiece', 'onepiece(1)', 'onepiece(2)', 'onepiece(3)', 'onepiece(4)']


Code nộp LeetCode (Cách 3):

In [11]:
class SolutionDict:
    def getFolderNames(self, names: list[str]) -> list[str]:
        d = {}
        ans = []
        for nm in names:
            if nm not in d:
                d[nm] = 1
                ans.append(nm)
                continue
            k = d[nm]
            while f"{nm}({k})" in d:
                k += 1
            new = f"{nm}({k})"
            d[nm] = k + 1
            d[new] = 1
            ans.append(new)
        return ans

print(SolutionDict().getFolderNames(["onepiece", "onepiece(1)", "onepiece(2)", "onepiece(3)", "onepiece"]))

['onepiece', 'onepiece(1)', 'onepiece(2)', 'onepiece(3)', 'onepiece(4)']


**Độ phức tạp (khấu hao):** mỗi lần `k += 1` là một lần "nhảy qua" một tên `nm(k)` **đã có**. Một tên `s` cụ thể có dạng `gốc(k)` cho **đúng một** cặp (gốc, k) (bóc cặp ngoặc cuối cùng) ⇒ nó chỉ có thể bị nhảy qua bởi tên gốc đó, và sau khi nhảy qua thì `d[gốc]` đã vượt `k` vĩnh viễn ⇒ mỗi tên bị nhảy qua **tối đa một lần**. Tổng số lần nhảy ≤ số tên trong `d` ≤ `2n`. Vậy tổng thời gian `O(n · L)`, bộ nhớ `O(n · L)`.

### Cách 4 — Biến thể: `set` + `dict` riêng (dễ đọc hơn cho người mới)

Tách hai vai trò: `used` (set các tên đã dùng) và `nxt` (dict `k` kế tiếp, mặc định 1 nhờ `defaultdict`). Logic y hệt Cách 3.

In [12]:
def folder_names_two_maps(names):
    used = set()
    nxt = defaultdict(lambda: 1)
    ans = []
    for nm in names:
        res = nm
        if nm in used:
            k = nxt[nm]
            while f"{nm}({k})" in used:
                k += 1
            nxt[nm] = k + 1
            res = f"{nm}({k})"
        used.add(res)
        ans.append(res)
    return ans

class SolutionTwoMaps:
    def getFolderNames(self, names: list[str]) -> list[str]:
        return folder_names_two_maps(names)

for names, want in examples:
    print(folder_names_two_maps(names) == want)

True
True
True


Ở Cách 4 ta không cần đặt `nxt[res] = 1` vì `defaultdict` tự trả 1 cho tên chưa gặp. Độ phức tạp như Cách 3.

## 8. TRACE từng bước

Trace Cách 3 trên `["a", "a(1)", "a", "a", "a(1)", "a(2)", "a"]` — đầu vào này có cả tên nhập trùng với tên sinh ra.

Dự kiến:

| i | nm | trong d? | k bắt đầu → kết thúc | ans | cập nhật d |
|---|---|---|---|---|---|
| 0 | a | không | — | a | a:1 |
| 1 | a(1) | không | — | a(1) | a(1):1 |
| 2 | a | có | 1 → 2 (a(1) chiếm) | a(2) | a:3, a(2):1 |
| 3 | a | có | 3 → 3 | a(3) | a:4, a(3):1 |
| 4 | a(1) | có | 1 → 1 | a(1)(1) | a(1):2, a(1)(1):1 |
| 5 | a(2) | có | 1 → 1 | a(2)(1) | a(2):2, a(2)(1):1 |
| 6 | a | có | 4 → 4 | a(4) | a:5, a(4):1 |

In [13]:
def trace_dict(names):
    d = {}
    ans = []
    for i, nm in enumerate(names):
        if nm not in d:
            d[nm] = 1
            ans.append(nm)
            print(f"i={i} nm={nm:8} | mới -> {nm:10} | d[{nm}]=1")
            continue
        k0 = k = d[nm]
        skipped = []
        while f"{nm}({k})" in d:
            skipped.append(f"{nm}({k})")
            k += 1
        new = f"{nm}({k})"
        d[nm] = k + 1
        d[new] = 1
        ans.append(new)
        print(f"i={i} nm={nm:8} | trùng, k: {k0}->{k}, nhảy qua {skipped} -> {new:10} | d[{nm}]={k+1}, d[{new}]=1")
    print("ans =", ans)
    return ans

_ = trace_dict(["a", "a(1)", "a", "a", "a(1)", "a(2)", "a"])

i=0 nm=a        | mới -> a          | d[a]=1
i=1 nm=a(1)     | mới -> a(1)       | d[a(1)]=1
i=2 nm=a        | trùng, k: 1->2, nhảy qua ['a(1)'] -> a(2)       | d[a]=3, d[a(2)]=1
i=3 nm=a        | trùng, k: 3->3, nhảy qua [] -> a(3)       | d[a]=4, d[a(3)]=1
i=4 nm=a(1)     | trùng, k: 1->1, nhảy qua [] -> a(1)(1)    | d[a(1)]=2, d[a(1)(1)]=1
i=5 nm=a(2)     | trùng, k: 1->1, nhảy qua [] -> a(2)(1)    | d[a(2)]=2, d[a(2)(1)]=1
i=6 nm=a        | trùng, k: 4->4, nhảy qua [] -> a(4)       | d[a]=5, d[a(4)]=1
ans = ['a', 'a(1)', 'a(2)', 'a(3)', 'a(1)(1)', 'a(2)(1)', 'a(4)']


Đếm số lần "nhảy qua": chỉ **1 lần** trong cả 7 tên (ở `i = 2`). Đó là trực giác của phân tích khấu hao.

Trace thêm Ví dụ 3 để thấy bước nhảy dài một lần duy nhất:

In [14]:
_ = trace_dict(examples[2][0] + ["onepiece", "onepiece(4)"])

i=0 nm=onepiece | mới -> onepiece   | d[onepiece]=1
i=1 nm=onepiece(1) | mới -> onepiece(1) | d[onepiece(1)]=1
i=2 nm=onepiece(2) | mới -> onepiece(2) | d[onepiece(2)]=1
i=3 nm=onepiece(3) | mới -> onepiece(3) | d[onepiece(3)]=1
i=4 nm=onepiece | trùng, k: 1->4, nhảy qua ['onepiece(1)', 'onepiece(2)', 'onepiece(3)'] -> onepiece(4) | d[onepiece]=5, d[onepiece(4)]=1
i=5 nm=onepiece | trùng, k: 5->5, nhảy qua [] -> onepiece(5) | d[onepiece]=6, d[onepiece(5)]=1
i=6 nm=onepiece(4) | trùng, k: 1->1, nhảy qua [] -> onepiece(4)(1) | d[onepiece(4)]=2, d[onepiece(4)(1)]=1
ans = ['onepiece', 'onepiece(1)', 'onepiece(2)', 'onepiece(3)', 'onepiece(4)', 'onepiece(5)', 'onepiece(4)(1)']


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def getFolderNames(self, names: list[str]) -> list[str]:
        d = {}                      # tên đã dùng -> k nên thử tiếp theo
        ans = []
        for nm in names:
            if nm in d:
                k = d[nm]
                while f"{nm}({k})" in d:
                    k += 1
                d[nm] = k + 1
                nm = f"{nm}({k})"
            d[nm] = 1
            ans.append(nm)
        return ans

for names, want in examples:
    assert Solution().getFolderNames(names) == want
print("OK ví dụ đề")

OK ví dụ đề


Chú ý mẹo ngắn gọn: sau khi tìm được `k`, ta gán đè `nm = nm(k)` rồi dùng chung hai dòng cuối `d[nm] = 1; ans.append(nm)` cho cả hai trường hợp.

## 10. Vì sao thuật toán đúng

Gọi `U_i` là tập tên đã cấp trước phút `i` (= các khoá của `d` lúc đó). Ta chứng minh bằng quy nạp hai điều:

**(A) Khoá của `d` đúng bằng `U_i`.** Mỗi bước thêm đúng một khoá là tên vừa cấp (`d[nm] = k + 1` chỉ đổi giá trị của khoá **đã có**).

**(B) Invariant:** với mọi khoá `x` trong `d`, các tên `x(1), …, x(d[x] − 1)` đều thuộc `U_i`.
- Khi tạo khoá mới `d[x] = 1`: khoảng rỗng, đúng.
- Khi xử lý trùng tên `x`: ta bắt đầu từ `k = d[x]` (mọi `j < d[x]` đã bị chiếm, theo invariant), tăng `k` chỉ khi `x(k)` bị chiếm ⇒ dừng ở `k` nhỏ nhất `≥ d[x]` còn trống ⇒ cũng là `k` nhỏ nhất `≥ 1` còn trống. Sau đó `x(k)` bị chiếm, nên đặt `d[x] = k + 1` vẫn giữ invariant.
- Tập `U` chỉ lớn lên nên tên nào đã bị chiếm thì mãi bị chiếm ⇒ invariant không bị phá ở các bước khác.

Từ (A) và (B): mỗi `ans[i]` là `names[i]` nếu chưa dùng, ngược lại là `names[i](k)` với `k` nhỏ nhất còn trống — đúng định nghĩa đề.

## 11. Phân tích Time / Space

Gọi `n = len(names)`, `L` = độ dài tối đa của tên (≤ 20, tên sinh ra dài thêm vài ký tự cho `(k)`).

| Cách | Time tệ nhất | Space |
|---|---|---|
| 1. list + k từ 1 | `O(n³ · L)` | `O(n · L)` |
| 2. set + k từ 1 | `O(n² · L)` (ví dụ `["a"] * n`) | `O(n · L)` |
| 3. dict nhớ k | `O(n · L)` khấu hao | `O(n · L)` |
| 4. set + dict nhớ k | `O(n · L)` khấu hao | `O(n · L)` |

Mỗi thao tác băm chuỗi tốn `O(L)`; tra `dict/set` là `O(1)` trung bình sau khi băm. Tổng số lần `k += 1` ở Cách 3 ≤ `2n` (mục 7).

## 12. Bộ test

### 12.1 Ví dụ đề + edge case (đối chiếu brute force)

In [16]:
ALL = {
    "brute": folder_names_brute,
    "set": folder_names_set,
    "dict": folder_names_dict,
    "two_maps": folder_names_two_maps,
    "sub_set": lambda a: SolutionSet().getFolderNames(a),
    "sub_dict": lambda a: SolutionDict().getFolderNames(a),
    "sub_two_maps": lambda a: SolutionTwoMaps().getFolderNames(a),
    "submit": lambda a: Solution().getFolderNames(a),
}
edge = [
    (["a"], ["a"]),
    (["a", "a"], ["a", "a(1)"]),
    (["a(1)", "a", "a"], ["a(1)", "a", "a(2)"]),
    (["a", "a", "a(1)"], ["a", "a(1)", "a(1)(1)"]),
    (["a(1)(1)", "a(1)", "a(1)"], ["a(1)(1)", "a(1)", "a(1)(2)"]),
    (["kaido", "kaido(1)", "kaido", "kaido(1)"], ["kaido", "kaido(1)", "kaido(2)", "kaido(1)(1)"]),
    (["x(0)", "x", "x"], ["x(0)", "x", "x(1)"]),
    (["()", "()", "()(1)"], ["()", "()(1)", "()(1)(1)"]),
]
for names, want in edge:
    assert folder_names_brute(names) == want, (names, folder_names_brute(names))
for name, f in ALL.items():
    for names, want in examples + edge:
        assert f(names) == want, (name, names, f(names))
print("Mọi cách đúng với", len(examples + edge), "test (edge case đã kiểm chứng bằng brute force)")

Mọi cách đúng với 11 test (edge case đã kiểm chứng bằng brute force)


### 12.2 Kiểm tra tính hợp lệ độc lập

Hàm `is_valid(names, ans)` kiểm tra theo **định nghĩa**: các tên trong `ans` phân biệt; nếu `names[i]` chưa dùng thì `ans[i] == names[i]`; nếu đã dùng thì `ans[i] = names[i](k)` và mọi `names[i](j)` với `j < k` đều đã dùng, còn `names[i](k)` thì chưa.

In [17]:
def is_valid(names, ans):
    if len(ans) != len(names) or len(set(ans)) != len(ans):
        return False
    used = set()
    for nm, got in zip(names, ans):
        if nm not in used:
            if got != nm:
                return False
        else:
            if not (got.startswith(nm + "(") and got.endswith(")")):
                return False
            ks = got[len(nm) + 1:-1]
            if not ks.isdigit() or ks[0] == "0":
                return False
            k = int(ks)
            if f"{nm}({k})" in used or any(f"{nm}({j})" not in used for j in range(1, k)):
                return False
        used.add(got)
    return True

print(is_valid(*examples[1]), is_valid(["a", "a"], ["a", "a(2)"]))

True False


### 12.3 Ngẫu nhiên (seed cố định)

Sinh tên từ một "kho" nhỏ gồm tên gốc và tên có hậu tố sẵn (như `b(1)`, `b(2)(1)`) để tạo thật nhiều va chạm.

In [18]:
def random_names(rng, n):
    pool = ["a", "b", "ab"]
    out = []
    for _ in range(n):
        nm = rng.choice(pool)
        for _ in range(rng.choice([0, 0, 1, 2])):
            nm += f"({rng.randint(0, 3)})"
        out.append(nm)
    return out

rng = random.Random(1487)
for _ in range(1500):
    names = random_names(rng, rng.randint(1, 30))
    want = folder_names_brute(names)
    assert is_valid(names, want)
    for name, f in ALL.items():
        got = f(names)
        assert got == want, (name, names, got, want)
print("1500 test ngẫu nhiên: mọi cách khớp brute force và hợp lệ")

1500 test ngẫu nhiên: mọi cách khớp brute force và hợp lệ


### 12.4 Hiệu năng: test xấu nhất cho Cách 2

`["a"] * n`: Cách 2 thử tổng `~n²/2` giá trị `k`. Ta đo với `n = 3000` cho Cách 2 và `n = 5·10^4` (giới hạn đề) cho Cách 3, 4.

In [19]:
n_small = 3000
bad = ["a"] * n_small
t0 = time.perf_counter(); r2 = folder_names_set(bad); t2 = time.perf_counter() - t0
t0 = time.perf_counter(); r3 = folder_names_dict(bad); t3 = time.perf_counter() - t0
assert r2 == r3
print(f"n={n_small}: set+k từ 1: {t2:.3f}s | dict: {t3:.4f}s")

big = ["a"] * 25000 + [f"a({i})" for i in range(1, 25001)]
rng.shuffle(big)
for name in ["dict", "two_maps", "submit"]:
    t0 = time.perf_counter(); res = ALL[name](big); dt = time.perf_counter() - t0
    assert len(set(res)) == len(res)
    print(f"n=50000 {name:9}: {dt:.3f}s")

n=3000: set+k từ 1: 0.782s | dict: 0.0021s
n=50000 dict     : 0.027s
n=50000 two_maps : 0.026s
n=50000 submit   : 0.029s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Không ghi nhận tên **sinh ra**

Chỉ thêm `names[i]` vào `used`, quên thêm `name(k)` vừa tạo. Khi sau đó có người nhập đúng tên đã sinh, hệ thống không biết nó đã bị chiếm ⇒ **trùng tên**.

In [20]:
def wrong1(names):
    used, ans = set(), []
    for nm in names:
        res = nm
        if nm in used:
            k = 1
            while f"{nm}({k})" in used:
                k += 1
            res = f"{nm}({k})"
        used.add(nm)                  # SAI: phải add(res)
        ans.append(res)
    return ans

case = ["a", "a", "a(1)"]
print("sai :", wrong1(case), "| hợp lệ?", is_valid(case, wrong1(case)))
print("đúng:", Solution().getFolderNames(case))

sai : ['a', 'a(1)', 'a(1)'] | hợp lệ? False
đúng: ['a', 'a(1)', 'a(1)(1)']


Bản sai cấp `a(1)` hai lần. **Sửa:** luôn `used.add(res)` (tên thực tế được cấp).

### Lỗi 2 — Dùng **số lần xuất hiện** làm `k`

Nghĩ rằng "lần thứ `c` gặp `nm` thì đặt `nm(c)`". Sai khi `nm(c)` đã bị ai đó nhập trực tiếp từ trước.

In [21]:
def wrong2(names):
    cnt, ans = {}, []
    for nm in names:
        if nm in cnt:
            ans.append(f"{nm}({cnt[nm]})")   # SAI: không kiểm tra trống
            cnt[nm] += 1
        else:
            ans.append(nm)
            cnt[nm] = 1
    return ans

case = examples[1][0]
print("sai :", wrong2(case), "| hợp lệ?", is_valid(case, wrong2(case)))
print("đúng:", Solution().getFolderNames(case))

sai : ['gta', 'gta(1)', 'gta(1)', 'avalon'] | hợp lệ? False
đúng: ['gta', 'gta(1)', 'gta(2)', 'avalon']


`gta(1)` bị cấp hai lần. **Sửa:** phải **kiểm tra** `nm(k)` có trống không (vòng `while`).

### Lỗi 3 — Tự "bóc" hậu tố của tên nhập vào

Nghĩ rằng `gta(1)` là "phiên bản 1 của gta" nên chuẩn hoá về `gta` rồi đánh số tiếp. Đề không nói vậy: `pes(2019)` phải giữ nguyên.

In [22]:
import re
def wrong3(names):
    d, ans = {}, []
    for nm in names:
        base = re.sub(r"\(\d+\)$", "", nm)    # SAI: bóc hậu tố
        if base not in d:
            d[base] = 1
            ans.append(base)
        else:
            ans.append(f"{base}({d[base]})")
            d[base] += 1
    return ans

case = examples[0][0]
print("sai :", wrong3(case))
print("đúng:", Solution().getFolderNames(case))

sai : ['pes', 'fifa', 'gta', 'pes(1)']
đúng: ['pes', 'fifa', 'gta', 'pes(2019)']


`pes(2019)` bị biến thành `pes(1)`. **Sửa:** coi mỗi tên nhập là một chuỗi nguyên vẹn.

### Lỗi 4 — Quên đặt `d[new] = 1` ở Cách 3

Nếu không thêm tên mới vào `d`, tên đó không được coi là "đã dùng" ⇒ cùng hậu quả như Lỗi 1.

In [23]:
def wrong4(names):
    d, ans = {}, []
    for nm in names:
        if nm not in d:
            d[nm] = 1
            ans.append(nm)
            continue
        k = d[nm]
        while f"{nm}({k})" in d:
            k += 1
        d[nm] = k + 1                 # SAI: thiếu d[new] = 1
        ans.append(f"{nm}({k})")
    return ans

case = ["b", "b", "b(1)", "b(1)"]
print("sai :", wrong4(case), "| hợp lệ?", is_valid(case, wrong4(case)))
print("đúng:", Solution().getFolderNames(case))

sai : ['b', 'b(1)', 'b(1)', 'b(1)(1)'] | hợp lệ? False
đúng: ['b', 'b(1)', 'b(1)(1)', 'b(1)(2)']


### Lỗi 5 — Đúng nhưng TLE: luôn thử `k` từ 1

Cách 2 đúng nhưng với `["a"] * 50000` cần ~1,25·10⁹ lần thử ⇒ quá thời gian. **Sửa:** nhớ `k` kế tiếp (Cách 3).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cấp **số nhỏ nhất còn trống**, tập đã dùng **chỉ tăng**" ⇒ nhớ con trỏ `next` cho từng nhóm, không quay lại từ đầu (giống tìm `mex` tăng dần).
- Một `dict` vừa làm "set đã dùng" vừa lưu dữ liệu phụ ⇒ code ngắn.
- Luôn tự hỏi: "**output** của bước này có thể trở thành **input/xung đột** cho bước sau không?" ⇒ nếu có, phải ghi nhận nó.
- Phân tích khấu hao: đếm **tổng** số lần vòng `while` chạy trên cả bài, không nhân "tệ nhất mỗi bước × n".
- Test nhanh bằng các ca: tên nhập trùng tên sinh ra (`["a","a","a(1)"]`), tên có sẵn hậu tố (`"pes(2019)"`), `["a"] * n`.

## 15. Biến thể của bài

1. **Tên có phần mở rộng:** `report.txt` trùng ⇒ `report(1).txt` (chèn hậu tố trước dấu chấm cuối) — xem Bài 5.
2. **Định dạng hậu tố khác:** `name_1`, `name (1)` (có khoảng trắng như Windows), `name-copy-2`… — chỉ đổi hàm ghép chuỗi.
3. **Có thao tác xoá thư mục:** tập đã dùng có thể giảm ⇒ invariant "chỉ tăng" bị phá; cần cấu trúc khác cho mỗi tên gốc (ví dụ heap các `k` đã giải phóng + bộ đếm).
4. **LC 1500 — Design a File Sharing System:** cấp ID nhỏ nhất còn trống có tái sử dụng ⇒ min-heap.
5. **Cấp ID nhỏ nhất chưa dùng (mex):** LC 41 First Missing Positive, LC 2336 Smallest Number in Infinite Set.

In [24]:
# Minh hoạ biến thể 2: định dạng "name (k)" kiểu Windows
def folder_names_windows(names):
    d, ans = {}, []
    for nm in names:
        if nm in d:
            k = d[nm]
            while f"{nm} ({k})" in d:
                k += 1
            d[nm] = k + 1
            nm = f"{nm} ({k})"
        d[nm] = 1
        ans.append(nm)
    return ans

print(folder_names_windows(["New folder", "New folder", "New folder (1)", "New folder"]))

['New folder', 'New folder (1)', 'New folder (1) (1)', 'New folder (2)']


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. list + k từ 1 | Mô phỏng thẳng đề | `O(n³·L)` | `O(n·L)` | Chỉ để kiểm chứng (brute force) |
| 2. set + k từ 1 | Tra băm, vẫn thử lại từ đầu | `O(n²·L)` | `O(n·L)` | Dữ liệu ít trùng; dễ viết |
| 3. dict nhớ k | Một dict: đã dùng + k kế tiếp | `O(n·L)` khấu hao | `O(n·L)` | Bản nộp chuẩn |
| 4. set + defaultdict | Tách hai vai trò cho dễ đọc | `O(n·L)` khấu hao | `O(n·L)` | Khi muốn code rõ ràng hơn |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().getFolderNames(["a", "a", "a(1)", "a"])` trả về gì?

### Bài 2 (tính tay)
Chạy tay Cách 3 trên `["x(1)", "x", "x", "x(1)", "x"]`, ghi `d` sau từng bước và `ans`.

### Bài 3 (điền TODO)
Hoàn thiện `count_renamed(names)`: đếm số thư mục **bị đổi tên** (tức `ans[i] != names[i]`) mà **không** dựng mảng `ans` đầy đủ… (được dùng `dict` như Cách 3).

### Bài 4 (tự viết lại)
Viết lại Cách 3 nhưng **không dùng `continue`**, và trả thêm tổng số lần vòng `while` chạy (`folder_names_count_skips(names) -> (ans, skips)`).

### Bài 5 (mini challenge)
Viết `file_names_with_ext(names)`: nếu tên có dấu chấm thì hậu tố chèn trước **dấu chấm cuối cùng** (`"a.txt"` trùng ⇒ `"a(1).txt"`); tên không có dấu chấm xử lý như đề.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_renamed(names):
    # TODO: dùng dict d như Cách 3, mỗi khi tên trùng thì tăng bộ đếm
    pass

# Bài 4 — TODO
def folder_names_count_skips(names):
    # TODO: như Cách 3, đếm mỗi lần k += 1
    pass

# Bài 5 — TODO
def file_names_with_ext(names):
    # TODO: tách stem, ext bằng rfind("."); thử stem(k)ext
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `a` thứ hai ⇒ `a(1)`. (2) Nhập `a(1)` — đã bị chiếm ⇒ thành `a(1)(1)`. (3) `a` thứ ba: `d["a"] = 2`, `a(2)` trống ⇒ `a(2)`.

**Bài 2.** (1) `x(1)` mới, `x` mới. (2) `x` thứ hai: `k = 1`, `x(1)` bị chiếm ⇒ `k = 2` ⇒ `x(2)`, `d[x] = 3`. (3) `x(1)` thứ hai ⇒ `x(1)(1)`; `x` cuối: `k = 3` ⇒ `x(3)`.

**Bài 3.** (1) Bộ đếm `cnt = 0`. (2) Nhánh `nm in d` ⇒ `cnt += 1`. (3) Vẫn phải cập nhật `d` đầy đủ (vì tên sinh ra ảnh hưởng về sau).

**Bài 4.** (1) Dùng `if nm in d: ... ` rồi hai dòng chung. (2) `skips += 1` trong `while`. (3) Kiểm tra `skips ≤ 2n` trên dữ liệu ngẫu nhiên.

**Bài 5.** (1) `i = nm.rfind(".")`; nếu `i == -1` thì `stem, ext = nm, ""`. (2) Ứng viên: `f"{stem}({k}){ext}"`. (3) Khoá nhớ `k` nên là chính `nm` (tên đầy đủ).

## 19. Đáp án tham khảo

In [26]:
# Bài 1
ans1 = Solution().getFolderNames(["a", "a", "a(1)", "a"])
print(ans1)
assert ans1 == ["a", "a(1)", "a(1)(1)", "a(2)"]

# Bài 2
ans2 = trace_dict(["x(1)", "x", "x", "x(1)", "x"])
assert ans2 == ["x(1)", "x", "x(2)", "x(1)(1)", "x(3)"]

['a', 'a(1)', 'a(1)(1)', 'a(2)']
i=0 nm=x(1)     | mới -> x(1)       | d[x(1)]=1
i=1 nm=x        | mới -> x          | d[x]=1
i=2 nm=x        | trùng, k: 1->2, nhảy qua ['x(1)'] -> x(2)       | d[x]=3, d[x(2)]=1
i=3 nm=x(1)     | trùng, k: 1->1, nhảy qua [] -> x(1)(1)    | d[x(1)]=2, d[x(1)(1)]=1
i=4 nm=x        | trùng, k: 3->3, nhảy qua [] -> x(3)       | d[x]=4, d[x(3)]=1
ans = ['x(1)', 'x', 'x(2)', 'x(1)(1)', 'x(3)']


In [27]:
# Bài 3
def count_renamed_sol(names):
    d, cnt = {}, 0
    for nm in names:
        if nm in d:
            cnt += 1
            k = d[nm]
            while f"{nm}({k})" in d:
                k += 1
            d[nm] = k + 1
            nm = f"{nm}({k})"
        d[nm] = 1
    return cnt

# Bài 4
def folder_names_count_skips_sol(names):
    d, ans, skips = {}, [], 0
    for nm in names:
        if nm in d:
            k = d[nm]
            while f"{nm}({k})" in d:
                k += 1
                skips += 1
            d[nm] = k + 1
            nm = f"{nm}({k})"
        d[nm] = 1
        ans.append(nm)
    return ans, skips

rng = random.Random(19)
for _ in range(500):
    names = random_names(rng, rng.randint(1, 40))
    ref = Solution().getFolderNames(names)
    assert count_renamed_sol(names) == sum(a != b for a, b in zip(names, ref))
    got, skips = folder_names_count_skips_sol(names)
    assert got == ref and skips <= 2 * len(names)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [28]:
# Bài 5
def file_names_with_ext_sol(names):
    d, ans = {}, []
    for nm in names:
        if nm in d:
            i = nm.rfind(".")
            stem, ext = (nm, "") if i == -1 else (nm[:i], nm[i:])
            k = d[nm]
            while f"{stem}({k}){ext}" in d:
                k += 1
            d[nm] = k + 1
            nm = f"{stem}({k}){ext}"
        d[nm] = 1
        ans.append(nm)
    return ans

res5 = file_names_with_ext_sol(["a.txt", "a.txt", "a(1).txt", "a.txt", "b", "b", "x.tar.gz", "x.tar.gz"])
print(res5)
assert res5 == ["a.txt", "a(1).txt", "a(1)(1).txt", "a(2).txt", "b", "b(1)", "x.tar.gz", "x.tar(1).gz"]
assert file_names_with_ext_sol(["pes", "pes"]) == Solution().getFolderNames(["pes", "pes"])
print("Đáp án Bài 1–5: mọi assert đều qua")

['a.txt', 'a(1).txt', 'a(1)(1).txt', 'a(2).txt', 'b', 'b(1)', 'x.tar.gz', 'x.tar(1).gz']
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "cấp tên/ID nhỏ nhất còn trống, tập đã dùng chỉ tăng":**

```python
d = {}                          # đã dùng -> k kế tiếp nên thử
ans = []
for nm in names:
    if nm in d:
        k = d[nm]
        while f"{nm}({k})" in d:
            k += 1
        d[nm] = k + 1           # lần sau khỏi thử lại
        nm = f"{nm}({k})"
    d[nm] = 1                   # ghi nhận cả tên SINH RA
    ans.append(nm)
return ans
```

**Checklist trước khi submit:**
- [ ] Tên **sinh ra** cũng được đánh dấu đã dùng.
- [ ] Kiểm tra `nm(k)` **có trống thật không**, không suy từ số lần xuất hiện.
- [ ] Không bóc/chuẩn hoá hậu tố của tên nhập vào.
- [ ] Nhớ `k` kế tiếp cho từng tên gốc để tránh `O(n²)`.
- [ ] Thử các ca: `["a","a","a(1)"]`, `["pes(2019)"]`, `["a"] * 50000`.
- [ ] Đúng chữ ký: `getFolderNames(self, names: list[str]) -> list[str]`.